# ZIP-Code Income Shifts in Travis County

**Scenario:** A redistricting analyst needs to understand how income
distributions shifted across Travis County ZIP codes between tax years.
The IRS Statistics of Income (SOI) publishes ZIP-code-level income data
annually. The BLS QCEW provides employment context.

siege_utilities wraps both sources with URL builders and parse normalizers
so the analyst can focus on the question, not the data plumbing.

## What this shows

Use IRS SOI ZIP-code income data and BLS QCEW employment data with deterministic fixture CSVs. This notebook demonstrates URL construction, schema-on-read parsing, FIPS/ZIP normalization, and economic context joins without network access or user cache paths.


In [ ]:
from pathlib import Path
import tempfile

import pandas as pd

from siege_utilities.economic.irs.soi import IRSSOIFiles
from siege_utilities.economic.bls.qcew import QCEWFiles

fixture_dir = tempfile.TemporaryDirectory()
cache_dir = Path(fixture_dir.name)
soi = IRSSOIFiles(cache_dir=cache_dir / "irs")
qcew = QCEWFiles(cache_dir=cache_dir / "bls")

print(f"IRS 2020 TX URL: {soi.url_for(tax_year=2020, state_abbrev='TX')}")
print(f"Offline fixture cache root: {cache_dir.name} (temporary)")


## 1. IRS SOI: Schema-on-read normalization

A tiny fixture stands in for the downloaded IRS CSV. The parser keeps ZIP and state FIPS as strings so leading zeros and join keys survive CSV ingestion.


In [ ]:
irs_csv = cache_dir / "irs_fixture.csv"
irs_csv.write_text(
    "STATEFIPS,STATE,ZIPCODE,agi_stub,N1,A00100\n"
    "48,TX,78701,1,120,4500\n"
    "48,TX,78702,2,95,8200\n"
)

irs_df = soi.parse(irs_csv)
print(irs_df[["STATEFIPS", "ZIPCODE", "agi_stub", "N1", "A00100"]])
assert irs_df["STATEFIPS"].tolist() == ["48", "48"]
assert irs_df["ZIPCODE"].tolist() == ["78701", "78702"]


## 2. BLS QCEW: Employment context fixture

The BLS parser filters by year, quarter, state FIPS, and NAICS depth. This keeps the notebook deterministic while preserving the production call shape.


In [ ]:
qcew_csv = cache_dir / "qcew_fixture.csv"
qcew_csv.write_text(
    "area_fips,own_code,industry_code,agglvl_code,qtrly_estabs,month1_emplvl,month2_emplvl,month3_emplvl,total_qtrly_wages,avg_wkly_wage,year,qtr\n"
    "48000,0,54,50,100,1000,1010,1020,25000000,1900,2020,1\n"
    "48000,0,541,70,10,100,110,120,3000000,2100,2020,1\n"
    "06000,0,54,50,80,900,910,920,20000000,1700,2020,1\n"
)

qcew_df = qcew.parse(qcew_csv, year=2020, quarter=1, state_fips="48", naics_depth=2)
print(qcew_df[["area_fips", "industry_code", "qtrly_estabs", "avg_wkly_wage"]])
assert qcew_df["area_fips"].tolist() == ["48000"]
assert qcew_df["industry_code"].tolist() == ["54"]


## 3. Join narrative: provenance and comparable keys

The notebook keeps source/year/quarter fields visible. In production, ZIP-level income and state/industry employment are different grains; do not treat this as a row-level causal join. Use it as context with explicit provenance and grain labels.


In [ ]:
income_summary = pd.DataFrame({
    "source": ["IRS SOI fixture"],
    "year": [2020],
    "state_fips": [irs_df["STATEFIPS"].iloc[0]],
    "zip_count": [irs_df["ZIPCODE"].nunique()],
    "returns": [int(irs_df["N1"].sum())],
})
employment_summary = pd.DataFrame({
    "source": ["BLS QCEW fixture"],
    "year": [int(qcew_df["year"].iloc[0])],
    "quarter": [int(qcew_df["qtr"].iloc[0])],
    "state_fips": [qcew_df["area_fips"].str[:2].iloc[0]],
    "sector_count": [qcew_df["industry_code"].nunique()],
})

print(income_summary)
print(employment_summary)


## Key Patterns

- Use temp/fixture data in governed notebooks; live downloads belong behind explicit opt-in.
- Preserve ZIP/FIPS identifiers as strings during schema-on-read parsing.
- Keep provenance, year, quarter, and data grain visible before joining economic context.


## Related

- Source: `siege_utilities/economic/irs/soi.py`, `siege_utilities/economic/bls/qcew.py`
- Tests: `tests/test_irs_soi.py`, `tests/test_economic_irs_soi_errors.py`, `tests/test_economic_bls_qcew.py`, `tests/test_economic_errors.py`
- Notebook governance: `tests/test_notebook_hygiene.py`, `tests/test_notebooks.py`, `scripts/check_notebook_inventory.py`
